In [1]:
import ee

ee.Initialize(project="earth-engine-project-495404")

print("Earth Engine works in notebook!")

Earth Engine works in notebook!


In [2]:
%pip install --quiet earthengine-api pandas plotly

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.3.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import ee
import pandas as pd
import plotly.express as px

print("ee version:", ee.__version__)
print("pandas version:", pd.__version__)
print("plotly version:", px.__name__, "imported successfully")

ee version: 1.7.25
pandas version: 3.0.1
plotly version: plotly.express imported successfully


In [4]:
import ee

ee.Initialize(project="earth-engine-project-495404")

# Sanity check: ask Earth Engine to compute 7+8 on its servers
result = ee.Number(7).add(8).getInfo()
print("✅ Earth Engine connected from notebook")
print("Sanity check (7+8):", result)

✅ Earth Engine connected from notebook
Sanity check (7+8): 15


In [5]:
districts = (
    ee.FeatureCollection("FAO/GAUL/2015/level2")
    .filter(ee.Filter.eq("ADM0_NAME", "Bangladesh"))
)

district_count = districts.size().getInfo()
print(f"Bangladesh districts loaded: {district_count}")

Bangladesh districts loaded: 64


In [6]:
rangpur = districts.filter(ee.Filter.eq("ADM2_NAME", "Rangpur")).first()
rangpur_geom = rangpur.geometry()

area_km2 = rangpur_geom.area().divide(1e6).getInfo()
centroid = rangpur_geom.centroid().coordinates().getInfo()

print(f"Rangpur area: {area_km2:.1f} km^2")
print(f"Rangpur centroid (lon, lat): {centroid[0]:.4f}, {centroid[1]:.4f}")

Rangpur area: 2314.1 km^2
Rangpur centroid (lon, lat): 89.2392, 25.6437


In [7]:
end_date = ee.Date(pd.Timestamp.today().strftime("%Y-%m-%d"))
start_date = end_date.advance(-24, "month")

s2_raw = (
    ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
    .filterBounds(rangpur_geom)
    .filterDate(start_date, end_date)
    .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 60))
)

scene_count = s2_raw.size().getInfo()
print(f"Sentinel-2 scenes (raw, <60% cloud): {scene_count}")

Sentinel-2 scenes (raw, <60% cloud): 199


In [8]:
def mask_s2_clouds(image):
    qa = image.select("QA60")
    cloud_bit = 1 << 10
    cirrus_bit = 1 << 11
    mask = (
        qa.bitwiseAnd(cloud_bit).eq(0)
        .And(qa.bitwiseAnd(cirrus_bit).eq(0))
    )
    return image.updateMask(mask).divide(10000).copyProperties(
        image, ["system:time_start"]
    )

s2 = s2_raw.map(mask_s2_clouds)
print("Cloud-masked collection ready")

Cloud-masked collection ready


In [9]:
def add_ndvi(image):
    ndvi = image.normalizedDifference(["B8", "B4"]).rename("NDVI")
    return image.addBands(ndvi)

s2_ndvi = s2.map(add_ndvi)
print("NDVI band added")

NDVI band added


In [10]:
def monthly_ndvi(month_offset):
    month_start = end_date.advance(month_offset, "month")
    month_end = month_start.advance(1, "month")

    monthly_collection = (
        s2_ndvi
        .filterDate(month_start, month_end)
        .select("NDVI")
    )

    ndvi_value = ee.Algorithms.If(
        monthly_collection.size().gt(0),
        monthly_collection.mean().reduceRegion(
            reducer=ee.Reducer.mean(),
            geometry=rangpur_geom,
            scale=100,
            maxPixels=1e9,
        ).get("NDVI"),
        None,
    )

    return ee.Feature(None, {
        "date": month_start.format("YYYY-MM"),
        "ndvi": ndvi_value,
    })

monthly_features = ee.FeatureCollection(
    [monthly_ndvi(i) for i in range(-23, 1)]
)

print("Monthly composites defined (24 months, empty-month safe)")

Monthly composites defined (24 months, empty-month safe)


In [11]:
features = monthly_features.getInfo()["features"]

rows = [
    (f["properties"]["date"], f["properties"].get("ndvi"))
    for f in features
]

df = pd.DataFrame(rows, columns=["date", "ndvi"])
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date").reset_index(drop=True)

print(df)
print(f"\nNon-null months: {df['ndvi'].notna().sum()} / {len(df)}")

         date      ndvi
0  2024-06-01  0.285853
1  2024-07-01  0.340794
2  2024-08-01  0.469642
3  2024-09-01  0.620434
4  2024-10-01  0.514139
5  2024-11-01  0.419894
6  2024-12-01  0.347733
7  2025-01-01  0.393950
8  2025-02-01  0.438343
9  2025-03-01  0.548528
10 2025-04-01  0.552286
11 2025-05-01  0.518903
12 2025-06-01  0.366124
13 2025-07-01  0.395305
14 2025-08-01  0.414239
15 2025-09-01  0.647687
16 2025-10-01  0.672404
17 2025-11-01  0.500942
18 2025-12-01  0.358661
19 2026-01-01  0.406869
20 2026-02-01  0.483747
21 2026-03-01  0.464122
22 2026-04-01  0.487753
23 2026-05-01       NaN

Non-null months: 23 / 24


In [12]:
%pip install --quiet nbformat

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.3.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [13]:
import plotly.graph_objects as go

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df["date"],
    y=df["ndvi"],
    mode="lines+markers",
    name="NDVI",
    line=dict(color="#2E7D32", width=2),
    marker=dict(size=6),
    connectgaps=False,
))

fig.update_layout(
    title="Rangpur District — Monthly Mean NDVI (Sentinel-2)",
    xaxis_title="Month",
    yaxis_title="NDVI",
    yaxis=dict(range=[0, 1]),
    template="simple_white",
    height=420,
    margin=dict(l=60, r=30, t=60, b=50),
)

fig.show()